# 08 — Quantile Regression

![Level](https://img.shields.io/badge/level-intermediate-yellow)
![Python](https://img.shields.io/badge/python-3.10%2B-blue)
![Twiga](https://img.shields.io/badge/twiga-forecast-orange)
![Time](https://img.shields.io/badge/time-~20%20min-lightgrey)

---

**What you'll build**

Prediction-interval forecasters using QR-LightGBM, QR-XGBoost, and FPQR-LightGBM, evaluated on PICP, NMPI, and Winkler score — giving you calibrated uncertainty estimates instead of point predictions.

**Prerequisites**
- [01 — Getting Started](01-getting-started.ipynb) (DataPipelineConfig, ForecasterConfig, TwigaForecaster.fit)
- [05 — ML Point Forecasting](05-ml-point-forecasting.ipynb) (gradient-boosted trees)
- [07 — Neural Networks](07-neural-networks.ipynb) (helpful but optional)
- Python: basic statistics, quartile concepts

**Learning objectives**

By the end of this notebook you will be able to:

1. Explain the pinball loss and why it trains models to predict specific quantiles
2. Configure `QRLIGHTGBMConfig`, `QRXGBOOSTConfig`, and `FPQRLIGHTGBMConfig` with custom quantile grids
3. Evaluate prediction intervals using PICP, NMPI, and Winkler score
4. Compare fixed-grid QR vs. FPQR and choose the right approach for your use case
5. Visualise quantile fans and interval coverage plots to diagnose calibration issues

## 1. Motivation — Why a Single Number Is Not Enough

A point forecast gives one number per time step: the model's best guess for net load at 14:30 tomorrow.  
For many operational decisions that single number is not enough.

**Energy dispatch example**  
A grid operator scheduling spinning reserves needs to know not just the expected net load but the *range of plausible outcomes*. If net load could realistically be 15 % higher than the median, reserves must cover that upside. Ignoring uncertainty leads to either:
- **under-reservation** — system reliability at risk, or  
- **over-reservation** — unnecessarily expensive.

**Prediction intervals** address this by providing a lower and upper bound that should contain the true value with a specified probability — e.g. a 90 % prediction interval should cover the actual observation 90 % of the time over many forecasts.

**Quantile Regression (QR)** is one of the most practical approaches:
- Train separate models (or a single multi-output model) for different quantile levels, e.g. q=0.05, 0.5, 0.95.
- The q=0.05 output is the lower bound and q=0.95 the upper bound for a 90 % interval.
- No distributional assumption is required — the model learns the conditional quantiles directly from data.
- Conformal post-processing can be applied to provide coverage guarantees.

> **Key concept — quantile regression and pinball loss**
>
> A standard regression model minimises the **mean squared error** (MSE), which targets the conditional mean of the output. A quantile regression model instead minimises the **pinball loss** (also called quantile loss) for a chosen quantile level τ ∈ (0, 1):
>
> ```
> L(y, q, τ) = τ · max(y − q, 0) + (1 − τ) · max(q − y, 0)
> ```
>
> Concretely:
> - When τ = 0.5, the pinball loss is equivalent to mean absolute error — the model targets the **median**.
> - When τ = 0.05, over-predictions are penalised 19× more than under-predictions — the model learns the **5th percentile** of the conditional distribution.
> - When τ = 0.95, under-predictions are penalised 19× more — the model learns the **95th percentile**.
>
> The pair (q₀.₀₅, q₀.₉₅) forms a **90 % prediction interval**. No distributional assumption (Gaussian, etc.) is required — the model learns the quantile structure directly from data.

> **Key concept — prediction interval evaluation metrics**
>
> Three complementary metrics assess whether a prediction interval is both **reliable** and **sharp**:
>
> | Metric | Full name | Formula (sketch) | Ideal value |
> |--------|-----------|-----------------|-------------|
> | **PICP** | Prediction Interval Coverage Probability | fraction of actuals inside [lower, upper] | ≥ 1 − α (e.g. ≥ 0.90) |
> | **NMPI** | Normalised Mean Prediction Interval | mean(upper − lower) / range(y) | as small as possible |
> | **Winkler score** | Winkler (1972) | width + penalty for misses | lower is better |
>
> - **PICP** measures reliability: is the interval wide enough to contain the true value at the claimed frequency? A 90 % interval that only captures 70 % of actuals is miscalibrated.
> - **NMPI** measures sharpness: are the bounds tight? An interval covering ±∞ has perfect PICP but is useless in practice.
> - **Winkler score** synthesises both: it equals the interval width when the actual is inside, and adds a large penalty when the actual falls outside. Lower is better, and it rewards narrow-but-accurate intervals over wide-but-safe ones.
>
> Raw QR bounds carry no formal coverage guarantee — empirical PICP may fall below `1 − α`. For a guarantee, apply conformal calibration (NB09).

> **Key concept — FPQR vs fixed-grid QR**
>
> Standard quantile regression uses a **fixed grid** of τ values specified at training time (e.g. 0.05, 0.10, …, 0.95). The model learns a separate output for each τ — an independent pinball loss is minimised for each level.
>
> **Flexible Proportional Quantile Regression (FPQR)** takes a different approach: a small `QuantileProposal` network learns to *propose* its own quantile grid adaptively, per sample and per horizon step. This has two practical consequences:
>
> 1. **Coverage shape adapts to the data** — if the conditional distribution is skewed or multimodal, FPQR can allocate more quantile mass where uncertainty is highest.
> 2. **Point forecast quality improves** — instead of reading off the median from a fixed grid, FPQR computes a weighted expectation ∑ τ̂ᵢ · qᵢ, which is a smoother and often more accurate point estimate.
>
> The trade-off: fixed QR lets you audit specific quantile levels (e.g. "what is the 5th percentile?") directly. FPQR returns an averaged grid that is harder to interpret at a specific level but better captures complex uncertainty shapes.
>
> **When to prefer FPQR**: signals with skewed residuals (solar irradiance, electricity prices), large datasets, or when interval shape matters more than auditability of specific quantile levels.

## 2. Setup

In [ ]:
import warnings

from great_tables import GT
from lets_plot import LetsPlot
import pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler

from twiga.core.plot.gt import twiga_gt

LetsPlot.setup_html()

from twiga.core.config import DataPipelineConfig, ForecasterConfig
from twiga.core.plot import (
    plot_forecast,
    plot_forecast_grid,
    plot_metrics_bar,
    plot_reliability_diagram,
)
from twiga.core.utils import configure, get_logger

warnings.filterwarnings("ignore")

configure()
log = get_logger("tutorials")

### Load data

In [ ]:
data = pd.read_parquet("../data/MLVS-PT.parquet")
data = data[["timestamp", "NetLoad(kW)", "Ghi", "Temperature"]]
data["timestamp"] = pd.to_datetime(data["timestamp"])
data = data.drop_duplicates(subset="timestamp").reset_index(drop=True)

log.info("Shape: %s", data.shape)
twiga_gt(GT(data.head().round(2)))

### Train / val / test splits

In [ ]:
from great_tables import GT, md

from twiga.core.plot.gt import twiga_gt

splits_df = pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Period": ["before 2021-01-01", "2021-01-01 – 2021-06-30", "2021-07-01 onwards"],
        "Purpose": ["Model learning", "Early-stopping / overfitting guard", "Final honest evaluation"],
    }
)

twiga_gt(
    GT(splits_df)
    .tab_header(title=md("**Data Splits**"), subtitle="Chronological — no shuffling, no overlap")
    .cols_label(**{c: md(f"**{c}**") for c in splits_df.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(splits_df),
)

In [ ]:
train_df = data[data["timestamp"] < "2021-01-01"].reset_index(drop=True)
val_df = data[(data["timestamp"] >= "2021-01-01") & (data["timestamp"] < "2021-07-01")].reset_index(drop=True)
test_df = data[data["timestamp"] >= "2021-07-01"].reset_index(drop=True)

log.info(
    f"train : {train_df.shape[0]:,} rows  ({train_df['timestamp'].min().date()} → {train_df['timestamp'].max().date()})"
)
log.info(f"val   : {val_df.shape[0]:,} rows  ({val_df['timestamp'].min().date()} → {val_df['timestamp'].max().date()})")
log.info(
    f"test  : {test_df.shape[0]:,} rows  ({test_df['timestamp'].min().date()} → {test_df['timestamp'].max().date()})"
)

### Shared pipeline and training configs

In [ ]:
data_config = DataPipelineConfig(
    target_feature="NetLoad(kW)",
    period="30min",
    latitude=32.371666,
    longitude=-16.274998,
    calendar_features=["hour", "day_night"],
    exogenous_features=["Ghi"],
    forecast_horizon=48,
    lookback_window_size=96,
    stride=48,
    input_scaler=StandardScaler(),
    target_scaler=RobustScaler(),
)

train_config = ForecasterConfig(split_freq="months", train_size=3, test_size=1)

data_config

## 3. ML Quantile Model — QR XGBoost

`QRXGBOOSTConfig` trains a multi-quantile XGBoost model. Each quantile level is a separate output; the median (`q=0.5`) becomes the point forecast and the outer quantiles (`q=0.05`, `q=0.95`) form a 90 % prediction band.

The interval emerges directly from the model — **no distributional assumption required**. For coverage-guaranteed intervals via conformal post-processing, see NB09 — Conformal Prediction.

In [ ]:
from IPython.display import clear_output

from twiga import TwigaForecaster
from twiga.models.ml import QRXGBOOSTConfig

qr_xg_config = QRXGBOOSTConfig(device="cpu")
forecaster_qr_xg = TwigaForecaster(
    data_params=data_config,
    model_params=[qr_xg_config],
    train_params=train_config,
)
forecaster_qr_xg.fit(train_df=train_df, val_df=val_df)
clear_output()
log.info("QR XGBoost training complete.")

**Reading QR XGBoost quantile metrics**

- **`pinball`** — the mean pinball loss averaged across all quantile levels and time steps. Lower is better; compare across models on the same dataset but do not compare across different `alpha` settings.
- **`sharpness`** — mean interval width (upper − lower). Smaller values indicate tighter, more informative intervals — desirable as long as PICP is maintained.
- **`mae` / `rmse`** — point forecast quality using the median quantile as the prediction. These match what you would see from a standard point-forecast model.

In [ ]:
pred_qr_xg, metric_qr_xg = forecaster_qr_xg.evaluate_quantile_forecast(test_df=test_df)
clear_output()
metric_qr_xg.columns

In [ ]:
from twiga.core.plot.gt import twiga_gt, twiga_report

In [ ]:
def get_metric_table(metric_df):
    res = (
        metric_df.groupby("Model")[["mae", "corr", "pinball", "crps", "sharpness", "calibration_error"]]
        .mean()
        .round(2)
        .reset_index()
    )
    res = res.rename(
        columns={
            "mae": "MAE",
            "corr": "Corr",
            "pinball": "PINBALL",
            "crps": "CRPS",
            "sharpness": "SHARPNESS",
            "calibration_error": "Cal-err",
        }
    )

    metric_name = ["MAE", "Corr", "PINBALL", "CRPS", "SHARPNESS", "Cal-err"]
    minimize_cols = ["MAE", "SMAPE", "RMSE"]
    maximize_cols = ["Corr"]

    return twiga_report(res, metric_name, minimize_cols, maximize_cols)

In [ ]:
get_metric_table(metric_qr_xg)

## 4. NN Quantile Model — MLPF QR

`MLPFConfig(distribution="qr")` adds a Group Additive Model head on top of the MLP encoder, giving the model an inductive bias toward additive feature contributions — useful when solar irradiance and calendar effects have near-separable impacts on net load.

As with the ML model, the quantile outputs form the interval directly. No calibration step is needed to obtain bounds.

In [ ]:
from twiga.models.nn import MLPFConfig

qr_mlpf_config = MLPFConfig(distribution="qr", max_epochs=10, rich_progress_bar=False)

forecaster_qr_mlpf = TwigaForecaster(
    data_params=data_config,
    model_params=[qr_mlpf_config],
    train_params=train_config,
)
forecaster_qr_mlpf.fit(train_df=train_df, val_df=val_df)
clear_output()
log.info("MLPGAM QR training complete.")

In [ ]:
pred_qr_mlpf, metric_qr_mlpf = forecaster_qr_mlpf.evaluate_quantile_forecast(test_df=test_df)
clear_output()

In [ ]:
get_metric_table(metric_qr_mlpf)

## 4b. NN Flexible-Quantile Model — MLPGAM FPQR

`MLPGAMFPQRConfig` pairs the GAM backbone with a **Flexible Proportional Quantile Regression** head.
Unlike fixed-grid QR (where `τ` values are pre-specified), FPQR learns its own quantile grid
adaptively during training via a `QuantileProposal` network.

**Key differences from fixed-grid QR:**

| Property | Fixed QR (`MLPGAMConfig(distribution="qr")`) | Flexible QR (FPQR) |
|---|---|---|
| Quantile levels | Fixed (e.g. 0.05 → 0.95) | Learned per sample |
| Grid size | Set by `quantiles` list | Set by `n_quantiles` |
| Point forecast | Median of fixed grid | Weighted expectation of proposed quantiles |
| Use case | Calibrated specific quantiles | Flexible uncertainty shape |

The `quantile_levels` returned are averaged over the batch for display purposes,
since each sample proposes a slightly different grid.

In [ ]:
fpqr_mlpf_config = MLPFConfig(
    distribution="fpqr",
    n_quantiles=9,
    conf_level=0.05,
    max_epochs=10,
    rich_progress_bar=False,
)

forecaster_fpqr_mlpf = TwigaForecaster(
    data_params=data_config,
    model_params=[fpqr_mlpf_config],
    train_params=train_config,
)
forecaster_fpqr_mlpf.fit(train_df=train_df, val_df=val_df)
clear_output()
log.info("MLPF FPQR training complete.")

In [ ]:
pred_fpqr_mlpf, metrics_fpqr_mlpf = forecaster_fpqr_mlpf.evaluate_quantile_forecast(test_df=test_df)
# clear_output()
# get_metric_table(metrics_fpqr_mlpf)

In [ ]:
from twiga.core.metrics import get_interval_metrics

pred_interval_fpqr_gam_iv, metrics_fpqr_gam_iv = forecaster_fpqr_gam.evaluate_interval_forecast(test_df=test_df)
clear_output()

interval_metrics_fpqr = get_interval_metrics(
    pred=pred_interval_fpqr_gam_iv["forecast"].values,
    true=pred_interval_fpqr_gam_iv["Actual"].values,
    lower=pred_interval_fpqr_gam_iv["lower"].values,
    upper=pred_interval_fpqr_gam_iv["upper"].values,
    alpha=0.1,
)
log.info("MLPGAM FPQR — interval metrics (alpha=0.10, target coverage=90%)")
log.info("\n%s", interval_metrics_fpqr.round(4).to_string(index=False))

In [ ]:
p_fpqr = plot_forecast(
    pred_interval_fpqr_gam_iv.iloc[: 7 * 48],
    actual_col="Actual",
    forecast_col="forecast",
    lower_col="lower",
    upper_col="upper",
    title="MLPGAM FPQR — 90% Prediction Interval (first 7 days of test)",
    y_label="NetLoad (kW)",
    x_label="Time step (30-min)",
)
p_fpqr

In [ ]:
pred_interval_qr_gam, metrics_interval_qr_gam = forecaster_qr_gam.evaluate_quantile_forecast(test_df=test_df)
clear_output()

log.info("MLPGAMQR interval result columns: %s", pred_interval_qr_gam.columns.tolist())
GT(pred_interval_qr_gam.head())

## 5. Interval Metrics

Three standard metrics assess prediction interval quality. Raw QR intervals do not carry a formal coverage guarantee — empirical PICP may differ from the nominal level. For guaranteed coverage, apply conformal calibration (NB09).

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

interval_metrics_df = pd.DataFrame(
    {
        "Metric": ["PICP", "NMPI", "Winkler score"],
        "Full name": [
            "Prediction Interval Coverage Probability",
            "Normalised Mean Prediction Interval",
            "Winkler (1972)",
        ],
        "Interpretation": [
            "Fraction of true values inside the interval. Should be ≥ 1 − alpha.",
            "Average interval width normalised by the target range. Smaller = sharper.",
            "Combines sharpness and coverage in one number. Lower is better.",
        ],
    }
)

twiga_gt(
    GT(interval_metrics_df)
    .tab_header(title=md("**Interval Evaluation Metrics**"), subtitle="Reliability, sharpness, and their synthesis")
    .cols_label(**{c: md(f"**{c}**") for c in interval_metrics_df.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(interval_metrics_df),
)

In [ ]:
from twiga.core.metrics import get_interval_metrics

interval_metrics_xg = get_interval_metrics(
    pred=pred_interval_qr_xg["forecast"].values,
    true=pred_interval_qr_xg["Actual"].values,
    lower=pred_interval_qr_xg["lower"].values,
    upper=pred_interval_qr_xg["upper"].values,
    alpha=0.1,
)
log.info("QR XGBoost — interval metrics (alpha=0.10, target coverage=90%)")
log.info("\n%s", interval_metrics_xg.round(4).to_string(index=False))

In [ ]:
model_results = {
    "QR XGBoost": pred_interval_qr_xg,
    "MLPGAMQR": pred_interval_qr_gam,
    "MLPGAMFPQR": pred_interval_fpqr_gam_iv,
}

rows = []
for name, df in model_results.items():
    m = get_interval_metrics(
        pred=df["forecast"].values,
        true=df["Actual"].values,
        lower=df["lower"].values,
        upper=df["upper"].values,
        alpha=0.1,
    )
    m.insert(0, "Model", name)
    rows.append(m)

comparison_df = pd.concat(rows, ignore_index=True).round(4)
log.info("Interval metric comparison — all models (alpha=0.10)")
comparison_df

**Reading the interval metric comparison**

- A model with **PICP < 1 − alpha** is under-covering — its intervals are too narrow and will miss more events than claimed. For a 90 % interval (alpha = 0.10), PICP below 0.90 indicates miscalibration.
- Among models with adequate PICP, prefer the one with the **smallest NMPI** (sharpest bounds) and **lowest Winkler score**.
- If all models under-cover, the raw quantile bounds need conformal calibration (NB09) to meet the nominal guarantee.
- FPQR may show different PICP than fixed-grid QR because its quantile grid adapts per sample — check both coverage and sharpness before choosing.

## 6. Visualising Prediction Intervals

A shaded band plot is the standard way to communicate intervals to stakeholders. We plot 7 days (336 half-hour steps) of actuals, the median forecast, and the 90 % band.

In [ ]:
log.info("Available columns in interval result: %s", pred_interval_qr_xg.columns.tolist())

In [ ]:
p = plot_forecast(
    pred_interval_qr_xg.iloc[: 7 * 48],
    actual_col="Actual",
    forecast_col="forecast",
    lower_col="lower",
    upper_col="upper",
    title="QR XGBoost — 90% Prediction Interval (first 7 days of test)",
    y_label="NetLoad (kW)",
    x_label="Time step (30-min)",
)
p

### Side-by-side comparison across models

In [ ]:
n = 7 * 48

combined_interval_df = pd.concat(
    [df.iloc[:n].assign(Model=name) for name, df in model_results.items()],
    ignore_index=True,
)

p = plot_forecast_grid(
    combined_interval_df,
    actual_col="Actual",
    forecast_col="forecast",
    model_col="Model",
    n_samples_per_model=n,
    title="90% Prediction Intervals — first 7 days of test set",
    y_label="NetLoad (kW)",
)
p

## 7. Comparing Alpha Levels

The choice of `alpha` controls the trade-off between coverage and sharpness. We retrain QR XGBoost with each alpha and compare PICP and NMPI.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

alpha_guide_df = pd.DataFrame(
    {
        "alpha": [0.05, 0.10, 0.20],
        "Target coverage": ["95 %", "90 %", "80 %"],
        "Interpretation": [
            "Wide band, high safety margin — use when missing an event is very costly",
            "Standard operating interval for most energy dispatch decisions",
            "Narrower band, more aggressive — use in real-time spot markets or when sharpness matters more",
        ],
    }
)

twiga_gt(
    GT(alpha_guide_df)
    .tab_header(title=md("**Alpha Level Guide**"), subtitle="Coverage vs. sharpness trade-off")
    .cols_label(**{c: md(f"**{c}**") for c in alpha_guide_df.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(alpha_guide_df),
)

In [ ]:
alphas = [0.05, 0.10, 0.20]
alpha_rows = []

for alpha_val in alphas:
    qr_a = TwigaForecaster(
        data_params=data_config,
        model_params=[QRXGBOOSTConfig(device="cpu", alpha=alpha_val)],
        train_params=train_config,
    )
    qr_a.fit(train_df=train_df, val_df=val_df)
    clear_output()

    pred_a, _ = qr_a.evaluate_interval_forecast(test_df=test_df)
    clear_output()

    m = get_interval_metrics(
        pred=pred_a["forecast"].values,
        true=pred_a["Actual"].values,
        lower=pred_a["lower"].values,
        upper=pred_a["upper"].values,
        alpha=alpha_val,
    )
    m.insert(0, "alpha", alpha_val)
    m.insert(1, "target_coverage", f"{int((1 - alpha_val) * 100)}%")
    alpha_rows.append(m)

alpha_comparison = pd.concat(alpha_rows, ignore_index=True)
log.info("Coverage vs. sharpness across alpha levels — QR XGBoost")
alpha_comparison[["alpha", "target_coverage", "picp", "nmpi", "winkle-score"]].round(4)

In [ ]:
alpha_bar_df = alpha_comparison[["alpha", "target_coverage", "picp", "nmpi", "winkle-score"]].copy()
alpha_bar_df["Configuration"] = alpha_bar_df["target_coverage"]

p_picp = plot_metrics_bar(
    alpha_bar_df.rename(columns={"picp": "PICP", "Configuration": "Model"}),
    metric_col="PICP",
    model_col="Model",
    lower_is_better=False,
    title="Coverage (PICP) by Alpha — QR XGBoost",
    x_label="PICP",
    horizontal=False,
)
p_picp

### Key observations

- **PICP** may fall short of the nominal target (`1 - alpha`) because raw quantile bounds carry no formal coverage guarantee. For guaranteed coverage, apply conformal calibration (NB09).
- **NMPI** increases as `alpha` decreases (wider intervals for higher coverage) — the fundamental coverage-sharpness trade-off.
- **Winkler score** synthesises both. The best alpha is context-dependent: energy operators with high cost of misses favour lower alpha (95 % coverage); real-time spot-market participants may tolerate alpha=0.20 for sharper signals.

## Wrapping up

**What you did**
- [x] Explained the pinball loss and why it produces calibrated conditional quantiles without distributional assumptions
- [x] Trained QR-XGBoost and MLPGAM QR (fixed-grid) quantile models via the Twiga API
- [x] Trained MLPGAM FPQR with a learned quantile grid and compared it to fixed-grid QR
- [x] Evaluated interval quality with PICP, NMPI, and Winkler score
- [x] Explored the coverage–sharpness trade-off across alpha levels (0.05, 0.10, 0.20)
- [x] Visualised prediction intervals as shaded band plots, grids, and fan charts

**Key takeaways**

1. The pinball loss targets specific conditional quantiles; pairs of quantiles form prediction intervals with no distributional assumption needed.
2. PICP measures reliability (is the interval wide enough?); NMPI measures sharpness (is it tight?); Winkler score synthesises both — always report all three.
3. Raw QR intervals may under-cover (PICP < 1 − alpha). For a formal guarantee, apply conformal calibration (NB09).
4. FPQR adapts its quantile grid per sample, giving better uncertainty shape on skewed signals; fixed-grid QR is easier to audit at specific quantile levels.
5. Increasing alpha narrows the band (higher sharpness) but risks under-coverage — choose alpha based on the operational cost of missing events.

---

## What's next?

**[08 — Parametric Distributions](08-parametric-distributions.ipynb)** — Move from non-parametric quantile bounds to full distribution forecasting: Normal, Laplace, Gamma, and Beta heads learn their parameters directly, producing richer uncertainty estimates via negative log-likelihood training.

**[09 — Conformal Prediction](09-conformal-prediction.ipynb)** — Apply conformal calibration on top of any model (including QR) to obtain **formal coverage guarantees** — empirical PICP will meet or exceed `1 − alpha` by construction.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

# QR workflow summary
workflow_df = pd.DataFrame(
    {
        "Step": ["Choose a QR model", "Assemble forecaster", "Train", "Get interval predictions", "Compute metrics"],
        "API call": [
            "QRXGBOOSTConfig(), MLPGAMConfig(distribution='qr'), or MLPGAMConfig(distribution='fpqr')",
            "TwigaForecaster(data_params, model_params, train_params)",
            ".fit(train_df, val_df)",
            ".evaluate_interval_forecast(test_df)",
            "get_interval_metrics(pred, true, lower, upper, alpha)",
        ],
    }
)

twiga_gt(
    GT(workflow_df)
    .tab_header(title=md("**Quantile Regression Workflow**"), subtitle="End-to-end steps in Twiga")
    .cols_label(**{c: md(f"**{c}**") for c in workflow_df.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(workflow_df),
)

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

# Fixed QR vs FPQR comparison
comparison_df = pd.DataFrame(
    {
        "Property": ["Quantile levels", "Point forecast", "Key param", "Typical advantage"],
        "Fixed QR": [
            "Fixed at init (conf_level bounds + quantiles list)",
            "Median quantile of fixed grid",
            "quantiles=[0.1, ..., 0.9]",
            "Easier to audit specific levels; stable training",
        ],
        "FPQR": [
            "Learned per sample via QuantileProposal network",
            "Weighted expectation ∑ τ̂ᵢ · qᵢ",
            "n_quantiles=9",
            "Better uncertainty shape on complex / skewed data",
        ],
    }
)

twiga_gt(
    GT(comparison_df)
    .tab_header(
        title=md("**Fixed-Grid QR vs Flexible QR (FPQR)**"),
        subtitle="Choose based on auditability vs. adaptability needs",
    )
    .cols_label(**{c: md(f"**{c}**") for c in comparison_df.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(comparison_df),
)

In [ ]:
# ruff: noqa: E501, E701, E702
from IPython.display import HTML

_TEAL = "#107591"
_TEAL_MID = "#069fac"
_TEAL_LIGHT = "#e8f5f8"
_TEAL_BEST = "#d0ecf1"
_TEXT_DARK = "#2d3748"
_TEXT_MUTED = "#718096"
_WHITE = "#ffffff"

steps = [
    {
        "num": "06",
        "title": "Backtesting & Evaluation",
        "desc": "Rolling-window backtesting · fold-level metrics",
        "tags": ["backtesting", "evaluation"],
        "active": False,
    },
    {
        "num": "07",
        "title": "Neural Networks",
        "desc": "MLPF · N-HiTS · Lightning training loop",
        "tags": ["neural network", "pytorch"],
        "active": False,
    },
    {
        "num": "08",
        "title": "Quantile Regression",
        "desc": "QR-LightGBM · QR-XGBoost · FPQR — calibrated prediction intervals",
        "tags": ["probabilistic", "quantile", "pinball loss"],
        "active": True,
    },
    {
        "num": "09",
        "title": "Parametric Distributions",
        "desc": "Normal · Laplace · Gamma heads — NLL training",
        "tags": ["parametric", "NLL", "distributions"],
        "active": False,
    },
    {
        "num": "10",
        "title": "Conformal Prediction",
        "desc": "Coverage-guaranteed intervals with CQR and CRC",
        "tags": ["conformal", "CQR", "CRC"],
        "active": False,
    },
]
track_name = "Probabilistic Track"
footer = 'Next: <span style="color:#107591;font-weight:600;">Parametric Distributions</span> (09) for NLL-trained models, or <span style="color:#107591;font-weight:600;">Conformal Prediction</span> (10) for coverage guarantees.'


def _b(t, bg, fg):
    return f'<span style="display:inline-block;background:{bg};color:{fg};font-size:10px;font-weight:600;padding:2px 7px;border-radius:10px;margin:2px 2px 0 0;">{t}</span>'


ch = ""
for i, s in enumerate(steps):
    a = s["active"]
    cb = _TEAL if a else _WHITE
    cbo = _TEAL if a else "#d1ecf1"
    nb = _TEAL_MID if a else _TEAL_LIGHT
    nf = _WHITE if a else _TEAL
    tf = _WHITE if a else _TEXT_DARK
    df = "#cce8ef" if a else _TEXT_MUTED
    bb = "#0d5f75" if a else _TEAL_BEST
    bf = "#b8e4ed" if a else _TEAL
    yh = (
        f'<span style="float:right;background:{_TEAL_MID};color:{_WHITE};font-size:10px;font-weight:700;padding:2px 10px;border-radius:12px;">★ you are here</span>'
        if a
        else ""
    )
    bdg = "".join(_b(t, bb, bf) for t in s["tags"])
    ch += f'<div style="background:{cb};border:2px solid {cbo};border-radius:12px;padding:16px 20px;display:flex;align-items:flex-start;gap:16px;box-shadow:{"0 4px 14px rgba(16,117,145,.25)" if a else "0 1px 4px rgba(0,0,0,.06)"};"><div style="min-width:44px;height:44px;background:{nb};color:{nf};border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:15px;font-weight:800;flex-shrink:0;">{s["num"]}</div><div style="flex:1;"><div style="font-size:15px;font-weight:700;color:{tf};margin-bottom:4px;">{s["title"]}{yh}</div><div style="font-size:12.5px;color:{df};margin-bottom:8px;line-height:1.5;">{s["desc"]}</div><div>{bdg}</div></div></div>'
    if i < len(steps) - 1:
        ch += f'<div style="display:flex;justify-content:center;height:32px;"><svg width="24" height="32" viewBox="0 0 24 32" fill="none"><line x1="12" y1="0" x2="12" y2="24" stroke="{_TEAL_MID}" stroke-width="2" stroke-dasharray="4 3"/><polygon points="6,20 18,20 12,30" fill="{_TEAL_MID}"/></svg></div>'

HTML(
    f'<div style="font-family:Inter,\'Segoe UI\',sans-serif;max-width:640px;margin:8px 0;"><div style="background:linear-gradient(135deg,{_TEAL} 0%,{_TEAL_MID} 100%);border-radius:12px 12px 0 0;padding:14px 20px;display:flex;align-items:center;gap:10px;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="{_WHITE}" stroke-width="2"><path d="M12 2L2 7l10 5 10-5-10-5z"/><path d="M2 17l10 5 10-5"/><path d="M2 12l10 5 10-5"/></svg><span style="color:{_WHITE};font-size:14px;font-weight:700;">Twiga Learning Path — {track_name}</span></div><div style="border:2px solid {_TEAL_LIGHT};border-top:none;border-radius:0 0 12px 12px;padding:20px 20px 16px;background:#f9fdfe;display:flex;flex-direction:column;">{ch}<div style="margin-top:16px;font-size:11.5px;color:{_TEXT_MUTED};text-align:center;border-top:1px solid {_TEAL_LIGHT};padding-top:12px;">{footer}</div></div></div>'
)

---
## Fan Chart

`plot_quantile_fan` renders graduated ribbon bands from widest to narrowest,
with the median line on top — the standard chart for communicating quantile
forecast uncertainty in energy and meteorological forecasting.

In [ ]:
from twiga.core.plot import plot_forecast_intervals, plot_quantile_fan

p_fan = plot_quantile_fan(
    pred_interval_qr_xg.iloc[: 7 * 48],
    actual_col="Actual",
    median_col="forecast",
    quantile_pairs=[("lower", "upper")],
    title="QR XGBoost — Fan Chart (90 % interval)",
    y_label="Net Load (kW)",
    fig_size=(820, 380),
)
p_fan

## Interval Comparison

`plot_forecast_intervals` overlays multiple models' prediction intervals as
semi-transparent ribbons — useful for visually comparing sharpness and coverage
across methods on the same axis.

In [ ]:
n_plot = 48  # one day

base_df = pd.DataFrame(
    {
        "Actual": pred_interval_qr_xg["Actual"].values[:n_plot],
        "xg_lo": pred_interval_qr_xg["lower"].values[:n_plot],
        "xg_hi": pred_interval_qr_xg["upper"].values[:n_plot],
        "xg_fc": pred_interval_qr_xg["forecast"].values[:n_plot],
        "gam_lo": pred_interval_qr_gam["lower"].values[:n_plot],
        "gam_hi": pred_interval_qr_gam["upper"].values[:n_plot],
        "gam_fc": pred_interval_qr_gam["forecast"].values[:n_plot],
        "fpqr_lo": pred_interval_fpqr_gam_iv["lower"].values[:n_plot],
        "fpqr_hi": pred_interval_fpqr_gam_iv["upper"].values[:n_plot],
        "fpqr_fc": pred_interval_fpqr_gam_iv["forecast"].values[:n_plot],
    }
)

interval_specs = [
    {"model": "QR XGBoost", "lower": "xg_lo", "upper": "xg_hi", "center": "xg_fc"},
    {"model": "MLPGAMQR", "lower": "gam_lo", "upper": "gam_hi", "center": "gam_fc"},
    {"model": "MLPGAMFPQR", "lower": "fpqr_lo", "upper": "fpqr_hi", "center": "fpqr_fc"},
]

p_cmp = plot_forecast_intervals(
    base_df,
    interval_specs=interval_specs,
    actual_col="Actual",
    title="QR XGBoost vs MLPGAM QR vs MLPGAM FPQR — 90 % Interval Comparison",
    y_label="Net Load (kW)",
    fig_size=(820, 380),
)
p_cmp